# 02 — Vanilla Diffusion Policy Baseline

이 노트북은 실행 순서만 정의합니다. 재현성/학습/샘플링/시각화 로직은 `pcdp/` 모듈에 있습니다.


## 1. Colab/local bootstrap


In [ ]:
# Common Colab/local bootstrap: locate the repo root and install from there.
import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = 'phase_conditioned_diffusion_policy'
COLAB_REPO_ROOT = Path('/content') / REPO_NAME
# If this notebook is opened in a fresh Colab runtime without running 00_colab_setup.ipynb,
# set PCDP_REPO_URL to your GitHub clone URL before this cell executes.
DEFAULT_REPO_URL = os.environ.get(
    'PCDP_REPO_URL',
    'https://github.com/YOUR_GITHUB_USERNAME/phase_conditioned_diffusion_policy.git',
)

def in_colab() -> bool:
    return 'google.colab' in sys.modules or Path('/content').exists() and 'COLAB_RELEASE_TAG' in os.environ

def find_repo_root(start: Path | None = None) -> Path | None:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / 'pyproject.toml').exists() and (path / 'pcdp').is_dir():
            return path
    if COLAB_REPO_ROOT.exists():
        return COLAB_REPO_ROOT
    return None

repo_root = find_repo_root()
if repo_root is None and in_colab():
    if 'YOUR_GITHUB_USERNAME' in DEFAULT_REPO_URL:
        raise RuntimeError(
            'Fresh Colab runtime detected but /content/phase_conditioned_diffusion_policy is missing. '
            'Run notebooks/00_colab_setup.ipynb first, or set the PCDP_REPO_URL environment '
            "variable to this repo\'s GitHub clone URL before running this cell."
        )
    subprocess.run(['git', 'clone', DEFAULT_REPO_URL, str(COLAB_REPO_ROOT)], check=True)
    repo_root = COLAB_REPO_ROOT

if repo_root is None:
    raise RuntimeError('Could not locate the repository root. Start Jupyter from the repo or run 00_colab_setup.ipynb in Colab.')

os.chdir(repo_root)
try:
    get_ipython().run_line_magic('cd', str(repo_root))
except NameError:
    pass

os.environ.setdefault('MUJOCO_GL', 'osmesa')
os.environ.setdefault('PYOPENGL_PLATFORM', 'osmesa')
if 'PCDP_ARTIFACT_ROOT' not in os.environ:
    default_artifact_root = Path('/content/pcdp_artifacts') if in_colab() else Path.home() / 'phase_conditioned_diffusion_policy_artifacts'
    os.environ['PCDP_ARTIFACT_ROOT'] = str(default_artifact_root)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=repo_root, check=True)
print(f'✓ repo root: {repo_root}')
print('✓ pcdp installed from repo root (editable)')
print(f'✓ artifact root: {os.environ["PCDP_ARTIFACT_ROOT"]}')


## 2. Artifact paths


In [ ]:
from pcdp.paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR, ensure_artifact_dirs
ensure_artifact_dirs()
print(f'✓ artifact root: {ARTIFACT_ROOT}')


## 3. Imports + Config

In [ ]:
import torch

from pcdp.configs import get_experiment_config
from pcdp.experiment_plots import plot_loss_curve
from pcdp.experiment_runner import (
    build_model,
    build_noise_scheduler,
    load_data_and_build_loaders,
    train_or_load_checkpoint,
)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}, device={device}')

cfg = get_experiment_config('vanilla')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()
print(f'Experiment config: {cfg.name} — {cfg.display_name}')


## 4. Data

In [ ]:
data, train_ds, val_ds, train_loader, val_loader = load_data_and_build_loaders(cfg, DATA_DIR)


## 5. Model + Scheduler

In [ ]:
model = build_model(cfg, data, device=device)
noise_scheduler, ns_config, NUM_INFERENCE_STEPS = build_noise_scheduler(cfg)
ema = cfg.build_ema(model)


## 6. Train or Load

In [ ]:
TRAIN = True
train_losses, val_log, best_ema_state, CKPT_PATH = train_or_load_checkpoint(
    train=TRAIN, cfg=cfg, model=model, ema=ema, noise_scheduler=noise_scheduler,
    train_loader=train_loader, val_loader=val_loader, checkpoints_dir=CHECKPOINTS_DIR, device=device,
)


## 7. Loss Curve

In [ ]:
plot_loss_curve(train_losses, val_log, FIGURES_DIR / cfg.artifacts.loss_plot_name, title=cfg.display_name)
